# Requirements

In [1]:
import polars as pl

Missing values in polars `Series` and `DataFrame`s are handled graceful.

The data file `data/missing_values.csv` has four columns, one with integer, floating point, categorical and string data respectively. Each column has one missing value.

# Representing missing values

When a file is read without any type specifications for the columns, polars will infer types.

In [2]:
data = pl.read_csv('data/missing_values.csv')

In [3]:
data

int_data,float_data,category_data,string_data
i64,f64,str,str
3,3.7,"""A""","""str1"""
5,5.3,"""A""","""str1_str1"""
7,7.5,"""B""","""str2_str1"""
17,-3.5,"""A""","""str1_str2"""
13,5.7,"""A""","""str2"""
23,7.1,"""B""","""str2_str2"""
null,5.5,"""A""","""str3"""
29,null,"""B""","""str3_str1"""
31,3.3,null,"""str2_str3"""


In [4]:
data.schema

Schema([('int_data', Int64),
        ('float_data', Float64),
        ('category_data', String),
        ('string_data', String)])

The numerical columns have the types you would expect, i.e., `pl.Int64` and `pl.Float64`.  The other columns are both of type `pl.String`.

For each column, a missing value is displayed, denoted by `null`.

In [5]:
data.null_count()

int_data,float_data,category_data,string_data
u32,u32,u32,u32
1,1,1,1


In [6]:
data.select(pl.sum_horizontal(pl.all().is_null()).alias('row_null_count'))

row_null_count
u32
0
0
0
0
0
0
1
1
1


# Computing and missing values

For numerical operations, such as column sum, the missing values are simply ignored.

In [7]:
data['int_data'].sum()

165

In [8]:
data['float_data'].sum()

42.3

In [9]:
data['int_data', 'float_data'].describe()

statistic,int_data,float_data
str,f64,f64
"""count""",9.0,9.0
"""null_count""",1.0,1.0
"""mean""",18.333333,4.7
"""std""",12.328828,3.442383
"""min""",3.0,-3.5
"""25%""",7.0,3.7
"""50%""",17.0,5.5
"""75%""",29.0,7.1
"""max""",37.0,7.7


`group_by` will include `null` as a group.

In [10]:
data['category_data', 'string_data'].describe()

statistic,category_data,string_data
str,str,str
"""count""","""9""","""9"""
"""null_count""","""1""","""1"""
"""mean""",null,null
"""std""",null,null
"""min""","""A""","""str1"""
"""25%""",null,null
"""50%""",null,null
"""75%""",null,null
"""max""","""B""","""str3_str1"""


In [11]:
data['category_data', 'string_data'] \
    .group_by('category_data') \
    .len()

category_data,len
str,u32
"""A""",5
null,1
"""B""",4


The missing value `null` is considered a group and counted as well.

You can drop these values.

In [12]:
data['category_data', 'string_data'] \
    .group_by('category_data') \
    .len() \
    .drop_nulls()

category_data,len
str,u32
"""A""",5
"""B""",4


# null versus NaN

Note that `null` represents missing data, which is distinct from a `nan` (Not a Number, NaN) floating point value.

In [13]:
data = data.with_columns(
    (pl.col('float_data')).sqrt().alias('sqrt_float')
)
data

int_data,float_data,category_data,string_data,sqrt_float
i64,f64,str,str,f64
3,3.7,"""A""","""str1""",1.923538
5,5.3,"""A""","""str1_str1""",2.302173
7,7.5,"""B""","""str2_str1""",2.738613
17,-3.5,"""A""","""str1_str2""",NaN
13,5.7,"""A""","""str2""",2.387467
23,7.1,"""B""","""str2_str2""",2.664583
null,5.5,"""A""","""str3""",2.345208
29,null,"""B""","""str3_str1""",null
31,3.3,null,"""str2_str3""",1.81659


The column `sqrt_float` has both a `nan` and a `null` value, the former because the square root of a negative number results in a NaN.

Missing data is typically ignored by aggregating functions, while NaNs are taken into account.

In [14]:
data['sqrt_float'].sum()

nan

In [15]:
data['sqrt_float'].drop_nans().sum()

18.953059353967273

Whereas `drop_nulls` an be used to explicitly drop rows that contain missing values, `drop_nans` can be used to drop rows that contains NaNs.

Polars makes the presense of NaNs very explicit so that they  can be handled appropriately.

# Selecting rows with missing data or NaNs

In [16]:
data.filter(pl.any_horizontal(pl.all().is_null()))

int_data,float_data,category_data,string_data,sqrt_float
i64,f64,str,str,f64
null,5.5,"""A""","""str3""",2.345208
29,null,"""B""","""str3_str1""",null
31,3.3,null,"""str2_str3""",1.81659
37,7.7,"""B""",null,2.774887


In [17]:
data.filter(pl.any_horizontal(pl.col('sqrt_float').is_nan()))

int_data,float_data,category_data,string_data,sqrt_float
i64,f64,str,str,f64
17,-3.5,"""A""","""str1_str2""",NaN
